# 💼 05 - 模拟交易

本 Notebook 演示：
1. 创建模拟账户
2. 手动买卖操作
3. 自动信号驱动交易
4. 查看账户状态

In [ ]:
import sys
sys.path.insert(0, '..')

from src.trading.paper_account import PaperAccount
from src.trading.risk_manager import RiskManager
from src.trading.signal_engine import SignalEngine
from src.trading.scheduler import TradingScheduler
from src.core.types import Side
from datetime import date

# 创建账户
account = PaperAccount(initial_capital=1_000_000.0)
rm = RiskManager(max_single_weight=0.30, max_daily_drawdown=0.10)
scheduler = TradingScheduler()

print(f'✅ 模拟账户已创建')
print(f'   初始资金: ¥{account.initial_capital:,.0f}')
print(f'   可用资金: ¥{account.cash:,.0f}')

## 1. 手动交易

In [ ]:
today = date.today()

# 手动买入
t1 = account.buy('000001.SZ', 11.00, 100, today)
if t1:
    print(f'✅ {t1.trade_id}: 买入 平安银行 x{t1.quantity} @ ¥{t1.price:.2f}')
    print(f'   佣金: ¥{t1.commission:.2f}')
    print(f'   剩余资金: ¥{account.cash:,.2f}')

# T+1: 当天不能卖出
t2 = account.sell('000001.SZ', 11.10, 100, today)
print(f'❌ 当日卖出: {"拒绝" if t2 is None else "成功"} (T+1限制)')

# 下一天可以卖
tomorrow = scheduler.next_trading_day(today)
account.update_t_plus_one(tomorrow)
print(f'\n📅 下一个交易日: {tomorrow}')

## 2. 自动信号驱动交易（历史回测模式）

In [ ]:
from src.data.aggregator import DataAggregator
from src.strategy.factors.momentum import MomentumFactor
from src.strategy.factors.trend import TrendFactor

# 获取数据
agg = DataAggregator()
my_stocks = ['600519.SH', '000001.SZ', '300750.SZ', '000858.SZ', '601318.SH']
bars = agg.get_bars_batch(my_stocks, date(2025, 4, 1), date.today())
dates = sorted(bars.index.get_level_values(0).unique())

# 策略配置
account2 = PaperAccount(initial_capital=1_000_000.0)
rm2 = RiskManager(max_single_weight=0.30)
engine = SignalEngine(top_n=3, rebalance_days=5, min_strength=0.3)
factors = [MomentumFactor(10), TrendFactor(5, 20)]

print(f'🔄 模拟交易: {len(dates)} 天, {len(my_stocks)} 只股票')

portfolio_history = []
for d in dates:
    account2.update_t_plus_one(d)
    rm2.reset_daily(d)

    try:
        prices = bars.loc[d]['close'].to_dict()
    except KeyError:
        continue

    # 生成信号
    historical = bars[bars.index.get_level_values(0) <= d]
    signals = engine.generate_signals(historical, factors, d)

    # 执行
    for sig in signals:
        price = prices.get(sig.symbol, 0)
        if price <= 0: continue
        risk = rm2.check_order(sig.symbol, sig.side.value, 100, price, account2, d)
        if not risk.passed: continue
        if sig.side == Side.BUY:
            account2.buy(sig.symbol, price, 100, d)
        else:
            account2.sell(sig.symbol, price, 100, d)

    pv = account2.portfolio_value(prices)
    portfolio_history.append({'date': d, 'value': pv})

print(f'✅ 模拟完成: {account2.summary()["total_trades"]} 笔交易')
print(f'   收益: {portfolio_history[-1]["value"] - account2.initial_capital:+,.2f}')
print(f'   市值: ¥{portfolio_history[-1]["value"]:,.2f}')

In [ ]:
import pandas as pd


In [ ]:
import matplotlib.pyplot as plt

df = pd.DataFrame(portfolio_history).set_index('date')
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df.index, df['value'], linewidth=2, color='steelblue')
ax.axhline(y=account2.initial_capital, color='gray', linestyle='--', alpha=0.5, label='初始资金')
ax.set_ylabel('组合价值 (¥)')
ax.set_title('💼 模拟交易组合价值')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. 保存账户状态

In [ ]:
path = account2.save('latest_paper.json')
print(f'💾 账户已保存: {path}')